In [1]:
# 저장된 모델의 가중치 데이터 불러오기
import torch
checkpoint= torch.load('/content/drive/MyDrive/MBCA/딥러닝 YOLO 심화과정/03 CNN 합성곱 신경망/01 Pytorch/Models/dog-breed-model.pth')

In [2]:
# 저장할때 2가지 정보를 저장했었음. model_state_dict, class_names
#1) 클래스 이름들
class_names= checkpoint['class_names']
print(class_names)

['affenpinscher', 'afghan_hound', 'african_hunting_dog', 'airedale', 'american_staffordshire_terrier', 'appenzeller', 'australian_terrier', 'basenji', 'basset', 'beagle', 'bedlington_terrier', 'bernese_mountain_dog', 'black-and-tan_coonhound', 'blenheim_spaniel', 'bloodhound', 'bluetick', 'border_collie', 'border_terrier', 'borzoi', 'boston_bull', 'bouvier_des_flandres', 'boxer', 'brabancon_griffon', 'briard', 'brittany_spaniel', 'bull_mastiff', 'cairn', 'cardigan', 'chesapeake_bay_retriever', 'chihuahua', 'chow', 'clumber', 'cocker_spaniel', 'collie', 'curly-coated_retriever', 'dandie_dinmont', 'dhole', 'dingo', 'doberman', 'english_foxhound', 'english_setter', 'english_springer', 'entlebucher', 'eskimo_dog', 'flat-coated_retriever', 'french_bulldog', 'german_shepherd', 'german_short-haired_pointer', 'giant_schnauzer', 'golden_retriever', 'gordon_setter', 'great_dane', 'great_pyrenees', 'greater_swiss_mountain_dog', 'groenendael', 'ibizan_hound', 'irish_setter', 'irish_terrier', 'iris

In [4]:
#2) 모델 가중치 데이터 정보
state_dict= checkpoint['model_state_dict']
#print(state_dict)   <--너무 길어서 일단...

In [5]:
# 가중치 데이터를 사용할 CNN분류모델 만들기
import torch.nn as nn
from torchvision import models
model= models.resnet152(weights=None)  # 모델의 구조만 가져오기. 가중치는 저장된 값으로 복원시킬 것임.
model.fc= nn.Linear(model.fc.in_features, 120)

In [6]:
# 위 구조만 있는 CNN 모델에 저장한 가중치값을 복원시키기!!
model.load_state_dict(state_dict)

<All keys matched successfully>

In [7]:
# GPU 환경으로 이동
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model= model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [8]:
# 여기까지가 모델 복원 완료!!!!

In [16]:
# 새로운 이미지를 불러와서 모델에게 품종분류 요청!
from PIL import Image
# 새로운 이미지도 모델에 맞게 전처리 해야하므로..
weight= models.ResNet152_Weights.DEFAULT
transform= weight.transforms()

# 새로운 이미지의 경로 준비 (말티즈)
image_path= "/content/drive/MyDrive/MBCA/딥러닝 YOLO 심화과정/03 CNN 합성곱 신경망/01 Pytorch/Test_Images/dog1.jpg"
image_path= "/content/drive/MyDrive/MBCA/딥러닝 YOLO 심화과정/03 CNN 합성곱 신경망/01 Pytorch/Test_Images/dog2.jpg"
image_path= "/content/drive/MyDrive/MBCA/딥러닝 YOLO 심화과정/03 CNN 합성곱 신경망/01 Pytorch/Test_Images/dog3.jpg"
image= Image.open(image_path).convert('RGB')
image= transform(image).unsqueeze(0).to(device)  # unsqueeze(): 3차원-->4차원 [c,h,w]-->[b,c,h,w]
#모델의 훈련데이터는 4차원 텐서 [배치사이즈, 채널, 높이, 너비]--[16,3,224,224]

In [17]:
#품종 예측
with torch.no_grad():
  outputs= model(image)
  probabilites= torch.softmax(outputs, dim=1)
  confidence, predicted= probabilites.max(dim=1)

In [18]:
#결과 출력
print('예측 클래스 번호:', predicted.item())
print('예측확률:', confidence.item() * 100)
print('예측 품종명:', class_names[predicted.item()])

예측 클래스 번호: 75
예측확률: 2.822783589363098
예측 품종명: miniature_pinscher
